In [ ]:
print("H")

In [ ]:
# Shared setup — imports, constants, paths
import os, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from facenet_pytorch import MTCNN
from scipy.fft import dctn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve,
)
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

# Backend-compatible constants — do not change
KYC_MAX_VIDEO_FRAMES = 5
KYC_FRAME_SIZE       = 224

ROOT            = Path("../../")
DATA_DIR        = ROOT / "data" / "kyc"
CROPS_DIR       = DATA_DIR / "crops"
WEIGHTS_DIR     = ROOT / "backend" / "weights"
PRECOMPUTED_DIR = ROOT / "backend" / "precomputed"
RESULTS_DIR     = Path("results")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("Device:", DEVICE)

In [ ]:
# Video-level 80/10/10 split from pre-extracted face crops
# Crop filenames: <source>_<videostem>_f<N>.jpg

def build_split(label):
    crops = list((CROPS_DIR / label).glob("*.jpg"))
    groups = {}
    for p in crops:
        key = p.stem.rsplit("_", 1)[0]
        groups.setdefault(key, []).append(p)
    videos = list(groups.keys())
    train_v, temp_v = train_test_split(videos, test_size=0.2, random_state=SEED)
    val_v,   test_v = train_test_split(temp_v, test_size=0.5, random_state=SEED)
    return {s: [p for v in vs for p in groups[v]]
            for s, vs in [("train",train_v),("val",val_v),("test",test_v)]}

real_splits = build_split("real")
fake_splits = build_split("fake")

split_dfs = {}
for split in ("train", "val", "test"):
    rows = [(str(p), 0) for p in real_splits[split]] + \
           [(str(p), 1) for p in fake_splits[split]]
    split_dfs[split] = pd.DataFrame(rows, columns=["path", "label"])
    n  = len(split_dfs[split])
    nr = (split_dfs[split].label == 0).sum()
    nf = (split_dfs[split].label == 1).sum()
    print(f"{split:5s}: {n} samples  (real={nr}, fake={nf})")

test_paths = split_dfs["test"]["path"].values

In [ ]:
# ImageNet normalization — matches backend/services/kyc/inference.py exactly
IMAGENET_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
AUGMENT_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FaceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform or IMAGENET_TRANSFORM
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self.transform(Image.open(row["path"]).convert("RGB")), int(row["label"])

def make_loaders(batch_size=32):
    train_dl = DataLoader(FaceDataset(split_dfs["train"], AUGMENT_TRANSFORM),
                          batch_size=batch_size, shuffle=True,  num_workers=2)
    val_dl   = DataLoader(FaceDataset(split_dfs["val"]),
                          batch_size=batch_size, shuffle=False, num_workers=2)
    test_dl  = DataLoader(FaceDataset(split_dfs["test"]),
                          batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl, test_dl

In [ ]:
# DCT feature extractor — matches backend/services/kyc/frequency_analyzer.py exactly
def extract_dct_features(face_image, size=224):
    gray    = face_image.convert("L").resize((size, size))
    arr     = np.array(gray, dtype=np.float32) / 255.0
    dct     = dctn(arr, norm="ortho")
    log_mag = np.log1p(np.abs(dct))
    log_mag = (log_mag - log_mag.min()) / (log_mag.max() - log_mag.min() + 1e-8)
    return log_mag[np.newaxis, :, :]   # shape: (1, H, W)

class DCTDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        dct = extract_dct_features(Image.open(row["path"]).convert("RGB"))
        return torch.tensor(dct, dtype=torch.float), int(row["label"])

def make_dct_loaders(batch_size=32):
    train_dl = DataLoader(DCTDataset(split_dfs["train"]), batch_size=batch_size, shuffle=True,  num_workers=2)
    val_dl   = DataLoader(DCTDataset(split_dfs["val"]),   batch_size=batch_size, shuffle=False, num_workers=2)
    test_dl  = DataLoader(DCTDataset(split_dfs["test"]),  batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl, test_dl

In [ ]:
# FrequencyCNN — matches backend/models/kyc/frequency_cnn.py exactly
class FrequencyCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),           nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),          nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))

In [ ]:
# Training utilities — shared by all experiments

def train_one_epoch(model, loader, optimizer, criterion):
    model.train(); total_loss = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

def evaluate_loader(model, loader):
    """Return (labels, fake_probs) arrays for a full DataLoader."""
    model.eval(); all_labels, all_probs = [], []
    with torch.no_grad():
        for x, y in loader:
            prob = F.softmax(model(x.to(DEVICE)), dim=1)[:,1].cpu().numpy()
            all_probs.extend(prob); all_labels.extend(y.numpy())
    return np.array(all_labels), np.array(all_probs)

def train_model(model, train_dl, val_dl, optimizer, scheduler,
                epochs=20, save_path=None, patience=5):
    """Train with early stopping on val ROC-AUC."""
    criterion = nn.CrossEntropyLoss()
    best_auc, wait = 0.0, 0
    history = {"train_loss": [], "val_auc": []}
    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, train_dl, optimizer, criterion)
        labels, probs = evaluate_loader(model, val_dl)
        val_auc = roc_auc_score(labels, probs)
        history["train_loss"].append(loss)
        history["val_auc"].append(val_auc)
        if scheduler: scheduler.step()
        print(f"Epoch {epoch:02d}/{epochs}  loss={loss:.4f}  val_auc={val_auc:.4f}")
        if val_auc > best_auc:
            best_auc, wait = val_auc, 0
            if save_path:
                torch.save(model.state_dict(), save_path)
                print(f"  -> Checkpoint saved (auc={best_auc:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}"); break
    return history

In [ ]:
# Standard 6-metric evaluation used by every experiment

def evaluate(y_true, y_pred_prob, threshold=0.5, title="Model"):
    y_pred = (np.array(y_pred_prob) >= threshold).astype(int)
    y_true = np.array(y_true)
    metrics = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":    recall_score(y_true, y_pred, zero_division=0),
        "F1":        f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC":   roc_auc_score(y_true, y_pred_prob),
        "PR-AUC":    average_precision_score(y_true, y_pred_prob),
    }
    print(f"\n--- {title} ---")
    for k, v in metrics.items(): print(f"  {k:12s}: {v:.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title)
    cm = confusion_matrix(y_true, y_pred)
    axes[0].imshow(cm, cmap="Blues")
    axes[0].set_title("Confusion Matrix")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, cm[i,j], ha="center", va="center", fontsize=14)
    axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
    axes[0].set_xticklabels(["Real","Fake"]); axes[0].set_yticklabels(["Real","Fake"])
    fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
    axes[1].plot(fpr, tpr, lw=2, label=f"AUC={metrics['ROC-AUC']:.3f}")
    axes[1].plot([0,1],[0,1],"--",color="grey")
    axes[1].set_title("ROC Curve"); axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
    axes[1].legend()
    prec, rec, _ = precision_recall_curve(y_true, y_pred_prob)
    axes[2].plot(rec, prec, lw=2, label=f"AP={metrics['PR-AUC']:.3f}")
    axes[2].set_title("PR Curve"); axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
    axes[2].legend()
    plt.tight_layout(); plt.show()
    return metrics

def threshold_analysis(y_true, y_pred_prob, title="Threshold Analysis"):
    thresholds = np.linspace(0.1, 0.9, 50)
    f1s, precs, recs = [], [], []
    for t in thresholds:
        yp = (np.array(y_pred_prob) >= t).astype(int)
        f1s.append(f1_score(y_true, yp, zero_division=0))
        precs.append(precision_score(y_true, yp, zero_division=0))
        recs.append(recall_score(y_true, yp, zero_division=0))
    plt.figure(figsize=(8,4))
    plt.plot(thresholds, f1s,   label="F1")
    plt.plot(thresholds, precs, label="Precision")
    plt.plot(thresholds, recs,  label="Recall")
    plt.axvline(0.50, color="orange", linestyle="--", label="suspicious (0.50)")
    plt.axvline(0.75, color="red",    linestyle="--", label="high_risk (0.75)")
    plt.title(title); plt.xlabel("Threshold"); plt.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# Display face crops with model score overlay
def show_examples(indices, probs, title, n=6):
    indices = list(indices[:n])
    if not indices: print(f"{title}: none"); return
    fig, axes = plt.subplots(1, len(indices), figsize=(3*len(indices), 3))
    if len(indices) == 1: axes = [axes]
    for ax, i in zip(axes, indices):
        ax.imshow(Image.open(test_paths[i])); ax.axis("off")
        ax.set_title(f"p={probs[i]:.2f}", fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

In [ ]:
# Experiment 2.3 — ViT-B/16
# Question: Does a transformer capture complementary evidence to CNNs?
# Backend artifact: weights/vit_best.pt

vit = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=2)

# Freeze all, then unfreeze last 4 transformer blocks + norm + head
for p in vit.parameters(): p.requires_grad = False
for block in vit.blocks[-4:]:
    for p in block.parameters(): p.requires_grad = True
for p in vit.norm.parameters(): p.requires_grad = True
for p in vit.head.parameters(): p.requires_grad = True

vit = vit.to(DEVICE)
print(f"Trainable params: {sum(p.numel() for p in vit.parameters() if p.requires_grad):,}")

In [ ]:
# Train — lr=5e-5, AdamW, CosineAnnealing, early stopping on val AUC
opt_vit   = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, vit.parameters()),
    lr=5e-5, weight_decay=1e-2)
sched_vit = torch.optim.lr_scheduler.CosineAnnealingLR(opt_vit, T_max=20)
VIT_CKPT  = WEIGHTS_DIR / "vit_best.pt"

train_dl, val_dl, test_dl = make_loaders(batch_size=32)

history_vit = train_model(
    vit, train_dl, val_dl,
    optimizer=opt_vit, scheduler=sched_vit,
    epochs=20, save_path=VIT_CKPT, patience=5,
)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_vit["train_loss"])
axes[0].set_title("ViT-B/16 — Train Loss"); axes[0].set_xlabel("Epoch")
axes[1].plot(history_vit["val_auc"])
axes[1].set_title("ViT-B/16 — Val ROC-AUC"); axes[1].set_xlabel("Epoch")
plt.tight_layout(); plt.show()

In [ ]:
# Load best checkpoint and evaluate
vit.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
labels_vit, probs_vit = evaluate_loader(vit, test_dl)
metrics_vit = evaluate(labels_vit, probs_vit, title="ViT-B/16")

In [ ]:
# Error overlap: does ViT make different mistakes than EfficientNet?
probs_eff  = np.load(RESULTS_DIR / "exp2.2_probs.npy")
labels_eff = np.load(RESULTS_DIR / "exp2.2_labels.npy")
preds_eff  = (probs_eff >= 0.5).astype(int)
preds_vit  = (probs_vit >= 0.5).astype(int)
eff_ok = preds_eff == labels_eff
vit_ok = preds_vit == labels_vit
print("Error overlap: EfficientNet vs ViT")
print(f"  Both correct      : {(eff_ok & vit_ok).sum()}")
print(f"  EfficientNet only : {(eff_ok & ~vit_ok).sum()}")
print(f"  ViT only          : {(~eff_ok & vit_ok).sum()}")
print(f"  Both wrong        : {(~eff_ok & ~vit_ok).sum()}")

In [ ]:
# Top disagreement samples (EfficientNet vs ViT)
disagreement = np.abs(probs_eff - probs_vit)
top_idx = np.argsort(disagreement)[::-1][:6]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
for ax, i in zip(axes, top_idx):
    ax.imshow(Image.open(test_paths[i])); ax.axis("off")
    true_label = "Fake" if labels_vit[i] else "Real"
    ax.set_title(f"True={true_label}\nEff={probs_eff[i]:.2f}  ViT={probs_vit[i]:.2f}", fontsize=7)
fig.suptitle("Top EfficientNet vs ViT Disagreements")
plt.tight_layout(); plt.show()

In [ ]:
# Save results + raw scores for exp2.5
prev = pd.read_csv(RESULTS_DIR / "exp2.2_results.csv")
results = pd.concat([prev, pd.DataFrame([{"Model":"ViT-B/16",
    **{k:round(v,4) for k,v in metrics_vit.items()}}])], ignore_index=True)
results.to_csv(RESULTS_DIR / "exp2.3_results.csv", index=False)
np.save(RESULTS_DIR / "exp2.3_labels.npy", labels_vit)
np.save(RESULTS_DIR / "exp2.3_probs.npy",  probs_vit)
print(results.to_string(index=False))

In [ ]:
# Verify backend artifact
size = VIT_CKPT.stat().st_size / 1024 / 1024
print(f"vit_best.pt: {size:.1f} MB")
print("exp2.3 done")